# Worksheet: Introduction to Recurrent Neural Networks

**Topic:** Recurrent Neural Networks, Sliding Windows, and Stock Forecasting

This interactive worksheet introduces sequence data, sliding windows, recurrent neuron outputs, and a simple RNN implementation using a stock-style dataset.

## Learning goals

By the end of this worksheet, students should be able to:

1. Explain why order matters in sequence data.
2. Create supervised learning examples using a sliding window.
3. Explain the meaning of window size and horizon.
4. Compute the output of a simple recurrent neuron.
5. Interpret the hidden state in an RNN.
6. Build and evaluate a simple RNN for stock-price forecasting.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

---

# 1. What is sequence data?

A **sequence** is an ordered collection of values. In sequence data, order matters.

Examples:
- daily stock prices
- weekly inventory
- monthly sales
- customer clickstream behavior
- words in a sentence

A standard feed-forward neural network treats observations as independent, but an RNN is designed to process observations over time.


If we use lagged values ($S_{t-5}, S_{t-4}, S_{t_3}, ..., S_{t-1}$ are predictors of $S_t$). RF, XGBoost, Feed-forward architectures assume that these values of S are independent. We need a model that accounts for trends, dependence for better predictions.

## Practice 1A: Concept check

1. Why is a stock-price series a sequence?  


**Your response here:**  
1.  Because data is ordered and the order matters.


---

# 2. Sliding window method

A sliding window turns a time series into a supervised learning dataset.

Example:

Series: `[10, 12, 13, 15, 18, 20]`  
Window size = 3  
Horizon = 1

| Input window | Target |
|---|---|
| [10, 12, 13] | 15 |
| [12, 13, 15] | 18 |
| [13, 15, 18] | 20 |

## Practice 2A: Sliding window by hand

Given the sequence:

`[100, 102, 101, 105, 107, 110]`

Use:
- window size = 3
- horizon = 1

Fill in the windows and targets.

| Input window | Target |
|---|---|
| [100, 102, 101] | 105 |
| [102, 101, 105] | 107 |
| [101, 105, 107] | 110 |


Use:
- window size = 3
- horizon = 2

Fill in the windows and targets.

| Input window | Target |
|---|---|
| [100, 102, 101] | 107 |
| [102, 101, 105] | 110 |


Use:
- window size = 4
- horizon = 1

Fill in the windows and targets.

| Input window | Target |
|---|---|
| [100, 102, 101, 105] | 107 |
| [102, 101, 105, 107] | 110 |


Tensorflow has a function called `TimeSeriesGenerator` that does this.

## Practice 2B: Code the sliding-window function

In [2]:
import numpy as np

def make_windows(series, window_size, horizon=1):
    X = []
    y = []

    for i in range(len(series) - window_size - horizon + 1):
        X.append(series[i : i + window_size])
        y.append(series[i + window_size + horizon - 1])

    return np.array(X), np.array(y)

In [3]:
# Self-check
series = np.array([100, 102, 101, 105, 107, 110])
X, y = make_windows(series, window_size=3, horizon=2)

print("X:")
print(X)
print("y:")
print(y)


X:
[[100 102 101]
 [102 101 105]]
y:
[107 110]


In [4]:
# Self-check
series = np.array([100, 102, 101, 105, 107, 110, 111, 112, 113])
X, y = make_windows(series, window_size=2, horizon=3)

print("X:")
print(X)
print("y:")
print(y)


X:
[[100 102]
 [102 101]
 [101 105]
 [105 107]
 [107 110]]
y:
[107 110 111 112 113]


In [5]:
# For RNNs and CNNs we use 3-way tensors: e.g. above X : (5 x 2), but we want X: (5 x 2 x 1)
# Input(shape = (5,2,1))
# For an RNN, reshape X to have shape:

X_rnn = X.reshape((X.shape[0], X.shape[1], 1))

print(X_rnn.shape)

(5, 2, 1)


## Practice 2C: Horizon

Suppose:

`[100, 102, 101, 105, 107, 110, 115]`

Window size = 3  
Horizon = 2

The first input window is `[100, 102, 101]`.

What is the target?

107 is the target for this input window.

In [7]:
# Self-check
series = np.array([100, 102, 101, 105, 107, 110, 115])
X_check, y_check = make_windows(series, window_size=3, horizon=2)

print("X:")
print(X_check)  ## X needs to be reshaped as a tensor for RNNs
print("y:")
print(y_check)  ## y is a vector y[0] is the target of the first obs

X:
[[100 102 101]
 [102 101 105]
 [101 105 107]]
y:
[107 110 115]


---

# 3. Multivariate sequences

A sequence can have more than one feature at each time step.

For example, a stock dataset might include:
- closing price
- daily return
- trading volume

RNN input shape is usually:

`(samples, time_steps, features)`

## Practice 3A: Shape interpretation

Suppose:
- 500 windows
- 20 time steps per window
- 3 features per time step

What is the RNN input shape?

In [8]:
(500, 20, 3)  ## for Input shape

(500, 20, 3)

## Practice 3B: Multivariate window function

In [9]:
def make_multivariate_windows(df, feature_cols, target_col, window_size, horizon=1):
    X = []
    y = []

    values = df[feature_cols].values
    target = df[target_col].values

    for i in range(len(values) - window_size - horizon + 1):
        X.append(values[i : i + window_size])
        y.append(target[i + window_size + horizon - 1])

    return np.array(X), np.array(y)

In [10]:
# Self-check for multivariate windows
data = {
    'price': [10, 11, 12, 13, 14, 15],
    'volume': [100, 110, 120, 130, 140, 150]
}

## For Price we would have
# [10, 11] ---> 12
# [11, 12] ---> 13
# ...............

## For Volume we would have
# [100, 110]
# [110, 120]
## ..............


## In multivariate we still have a single y (so we dont need to predict the price and the volume)
## (4 windows x 2 lags x 2 features)


df = pd.DataFrame(data)

# window_size=2, horizon=1, features=['price', 'volume'], target='price'
X_multi, y_multi = make_multivariate_windows(
    df,
    feature_cols=['price', 'volume'],
    target_col='price',
    window_size=2,
    horizon=1
)

print("Multivariate X shape:", X_multi.shape)
print("First window in X:\n", X_multi[0])
print("First target in y:", y_multi[0])

Multivariate X shape: (4, 2, 2)
First window in X:
 [[ 10 100]
 [ 11 110]]
First target in y: 12


In [11]:
X_multi

array([[[ 10, 100],
        [ 11, 110]],

       [[ 11, 110],
        [ 12, 120]],

       [[ 12, 120],
        [ 13, 130]],

       [[ 13, 130],
        [ 14, 140]]])

In [12]:
y_multi

array([12, 13, 14, 15])

---

# 4. Recurrent neuron output

A simple RNN neuron uses the current input and the previous hidden state.

$h_t = \tanh(w_x x_t + w_h h_{t-1} + b)$

where:
- $x_t$ is the current input
- $h_{t-1}$ is the previous hidden state
- $h_t$ is the new hidden state

## Practice 4A: One-step recurrent neuron

Suppose:

- $x_t = 2.0$   -------> a value at time t (current) of seq
- $h_{t-1} = 0.5$ -------> rolling weighted avg (at time t-1)
- $w_x = 0.8$  --------> weight of the current value
- $w_h = 0.4$  --------> weight to rolling weighted average
- $b = -0.1$  --------> bias (add or subract out of the total sum)

Compute:

$h_t = \tanh(w_x x_t + w_h h_{t-1} + b)$

In [15]:
## this is a single time step
x_t = 2.0
h_prev = 0.5 ## hidden state
w_x = 0.8
w_h = 0.4
b = -0.1

# TODO
h_t = np.tanh(0.8*2.0 + 0.4 * 0.5 - 0.1)

print("h_t =", h_t)

h_t = 0.935409070603099


In [14]:
ht = np.tanh(w_x * x_t + w_h * h_prev + b)
print(ht)

0.935409070603099


## Practice 4B: Recurrent neuron across a sequence

Use the sequence:

`[1.0, 2.0, -1.0]`

Initial hidden state = 0.

In [16]:
## step 1:
h = 0 ## hidden state
w_x = 0.8
w_h = 0.4
b = -0.1

h = np.tanh(w_x * 1.0 + w_h * h + b)
print("h_1 =", h)

h_1 = 0.6043677771171635


In [17]:
## step 2:

w_x = 0.8
w_h = 0.4
b = -0.1

h = np.tanh(w_x * 2.0 + w_h * h + b)
print("h_2 =", h)

h_2 = 0.9404289270915237


In [18]:
## step 3:

w_x = 0.8
w_h = 0.4
b = -0.1

h = np.tanh(w_x * -1.0 + w_h * h + b)
print("h_3 =", h)

h_3 = -0.4806493957651368


In [19]:
sequence = np.array([1.0, 2.0, -1.0])
h = 0.0
hidden_states = []

for x_val in sequence:
    h = np.tanh(w_x * x_val + w_h * h + b)
    hidden_states.append(h)

print("Hidden states:", hidden_states)

Hidden states: [np.float64(0.6043677771171635), np.float64(0.9404289270915237), np.float64(-0.4806493957651368)]


In [20]:
## if you need the whole seq: seq to seq
hidden_states

[np.float64(0.6043677771171635),
 np.float64(0.9404289270915237),
 np.float64(-0.4806493957651368)]

In [21]:
## if you need to predict the next day price: seq2vec
hidden_states[-1]

np.float64(-0.4806493957651368)

## Practice 4C: Interpretation

Why does the hidden state matter in an RNN?

**Your response here:**  
-

---

# 5. Fit an ML Regressors and RNN to stock data


---

# 6. Limitations of simple RNNs

Simple RNNs can struggle with:
- long-term dependencies
- vanishing gradients
- slow or unstable training

More advanced alternatives include:
- LSTM
- GRU
- attention-based models
- temporal CNNs

## Practice 6A

Why might a simple RNN struggle if the important pattern happened many time steps ago?

**Your response here:**  
-

---

# 7. Quiz-style review

Answer the following:

1. What is a sliding window?
2. What does horizon mean?
3. What is the hidden state in an RNN?
4. For RNN input shape `(samples, time_steps, features)`, what does each dimension mean?
5. Why should train/test splits for time series usually avoid random shuffling?
6. What is one limitation of a simple RNN?

**Your responses here:**  
1.  
2.  
3.  
4.  
5.  
6.

## Quick self-check

Suggested answers:
- A sliding window turns a sequence into supervised input-target examples.
- Horizon is how far ahead we predict.
- The hidden state carries information from previous time steps.
- Samples = number of windows, time_steps = length of each window, features = variables at each time step.
- Shuffling can leak future information into the training set.
- Simple RNNs can struggle with long-term dependencies and vanishing gradients.